In [10]:
import sys
sys.path.append("../src")

import os
import pandas as pd

import video_loader
import preprocessing

In [15]:
print("video_loader functions:", [x for x in dir(video_loader) if "video" in x or "scan" in x or "sample" in x])
print("preprocessing functions:", [x for x in dir(preprocessing) if "process" in x or "filter" in x or "blur" in x])

video_loader functions: ['get_video_metadata', 'sample_frames', 'scan_videos']
preprocessing functions: ['blur_score', 'filter_blurry_frames']


In [12]:
DATA_DIR = "../data"
OUTPUT_DIR = "../outputs/sampled_frames"
DOMAINS = ["artificial_jewellery", "home_cleaning", "shop"]

# Debug vs batch switches:
FULL_BATCH_MODE = True          # True = all videos, False = first video only
MAX_FRAMES_PER_VIDEO = None     # set e.g. 50 for quick tests
SAMPLE_RATE = 30                # every 30th frame
BLUR_THRESHOLD = 100            # tune later

In [18]:
video_records = video_loader.get_all_video_paths(DATA_DIR, DOMAINS)
df = pd.DataFrame(video_records)
print("Total videos found:", len(df))
df.head()

Total videos found: 10


,domain,video_name,video_path
0,artificial_jewellery,video_20260404_114752.mp4,../data/artificial_jewellery/video_20260404_11...
1,artificial_jewellery,video_20260404_120254.mp4,../data/artificial_jewellery/video_20260404_12...
2,artificial_jewellery,video_20260404_121005.mp4,../data/artificial_jewellery/video_20260404_12...
3,artificial_jewellery,video_20260404_115542.mp4,../data/artificial_jewellery/video_20260404_11...
4,artificial_jewellery,video_20260404_122845.mp4,../data/artificial_jewellery/video_20260404_12...


In [19]:
os.makedirs("../outputs", exist_ok=True)

if not video_records:
    raise ValueError("No videos found. Check folder names and video extensions.")

if FULL_BATCH_MODE:
    to_process = video_records
else:
    to_process = [video_records[0]]

logs = preprocessing.process_video_batch(
    to_process,
    output_dir=OUTPUT_DIR,
    sample_rate=SAMPLE_RATE,
    blur_threshold=BLUR_THRESHOLD,
    max_frames_per_video=MAX_FRAMES_PER_VIDEO
)

log_df = pd.DataFrame(logs)
log_df.to_csv("../outputs/preprocess_log.csv", index=False)
log_df

Batch preprocessing: 100%|██████████| 10/10 [03:21<00:00, 20.15s/it]


,domain,video_name,video_path,sample_rate,total_sampled_frames,clean_frames_saved,blur_threshold,status
0,artificial_jewellery,video_20260404_114752.mp4,../data/artificial_jewellery/video_20260404_11...,30,405,377,100,success
1,artificial_jewellery,video_20260404_120254.mp4,../data/artificial_jewellery/video_20260404_12...,30,312,79,100,success
2,artificial_jewellery,video_20260404_121005.mp4,../data/artificial_jewellery/video_20260404_12...,30,129,25,100,success
3,artificial_jewellery,video_20260404_115542.mp4,../data/artificial_jewellery/video_20260404_11...,30,405,224,100,success
4,artificial_jewellery,video_20260404_122845.mp4,../data/artificial_jewellery/video_20260404_12...,30,356,16,100,success
5,artificial_jewellery,video_20260405_105612_edit.mp4,../data/artificial_jewellery/video_20260405_10...,30,11,8,100,success
6,artificial_jewellery,video_20260404_121903.mp4,../data/artificial_jewellery/video_20260404_12...,30,457,427,100,success
7,home_cleaning,video_20260404_073018.mp4,../data/home_cleaning/video_20260404_073018.mp4,30,407,1,100,success
8,home_cleaning,video_20260404_074030.mp4,../data/home_cleaning/video_20260404_074030.mp4,30,258,3,100,success
9,shop,video_20260405_163219_edit.mp4,../data/shop/video_20260405_163219_edit.mp4,30,222,22,100,success


In [20]:
from pathlib import Path

p = Path(OUTPUT_DIR)
jpgs = list(p.rglob("*.jpg"))
print("Total saved frames:", len(jpgs))

# show example folder
if jpgs:
    print("Example frame:", jpgs[0])

Total saved frames: 1190
Example frame: ../outputs/sampled_frames/frame_210.jpg
